Import libraries

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#%matplotlib inline
import pandas as pd
import math
import numpy as np
from datetime import datetime, timedelta
from scipy import stats
import warnings
from IPython.display import display
import ipywidgets as widgets
pd.options.mode.chained_assignment = None 
from datetime import timezone
import datetime
from suntime import Sun
import pytz
from numpy.linalg import norm
from sklearn.metrics.pairwise import cosine_similarity

Read CSV

In [3]:
dataset = pd.read_csv("10sec-original.csv")

dataset['datetime'] = pd.to_datetime(dataset['datetime'], format='mixed')
lat = 18.47
lon = -66.12
tz = pytz.timezone('America/Puerto_Rico')

In [4]:
dataset

,tagID,datetime,event
0,1621,2017-06-21 08:00:10.850,entering
1,1621,2017-06-21 08:10:33.850,exiting
2,1621,2017-06-21 08:24:48.250,entering
3,1621,2017-06-21 08:51:23.450,entering
4,1621,2017-06-21 08:55:48.300,exiting
...,...,...,...
1855,661,2017-06-22 17:09:08.250,entering
1856,2258,2017-06-22 11:49:10.700,entering
1857,2258,2017-06-22 12:07:42.750,exiting
1858,2258,2017-06-22 16:14:06.350,entering


Data cleaning functions

Separate entry/exit events and convert into flights, takes entry/exit dataset
for a single bee and returns a table.

In [5]:

def classifyLoc(dataset):

    bee = dataset
    
    cur = bee['event']
    nxt = cur.shift(-1)
    cur_dt = bee['datetime']
    nxt_dt = cur_dt.shift(-1)
    #selects all except last
    valid = nxt.notna()
    location = np.select(
        [
            (cur == "entering") & (nxt == "exiting"),
            (cur == "entering") & (nxt == "exiting-entering"),
            
            (cur == "entering-exiting") & (nxt == "entering"),
            (cur == "exiting") & (nxt == "entering"),
            (cur == "exiting") & (nxt == "entering-exiting"),
            
            (cur == "entering") & (nxt == "entering"),
            (cur == "outside") & (nxt == "outside"),
        ],
        ["Inside", "Inside", "Outside", "Outside", "Outside", "Unknown-Inside", "Outside-Unknown"],
        default="Unknown",
    )
            
    t0 = cur_dt.iloc[0]
    age = (cur_dt - t0).dt.days

    #new dataset with inside/outside and time start/end
    BeeTravel = pd.DataFrame(
        {
            "location": location[valid],
            "start": cur_dt[valid],
            "end": nxt_dt[valid],
            "age": age[valid]
        }
    ).reset_index(drop=True)
    
    #classify short outside/insides as part of another group
    BeeTravel['activity'] = "undefined"
    BeeTravel.loc[BeeTravel["location"] == "Inside", "activity"] = "inside"
    duration = (BeeTravel["end"] - BeeTravel["start"]).dt.total_seconds()
    outside = BeeTravel["location"].isin(["Outside"])
   
    #Classify outside flights depending on length
    BeeTravel.loc[outside & (duration < 300), "activity"] = "outside-short"
    BeeTravel.loc[outside & (duration >= 300) & (duration  < 7200), "activity"] = "outside-foraging"
    BeeTravel.loc[outside & (duration >= 7200), "activity"] = "outside-long"
    BeeTravel['tagID'] = dataset['tagID'].iloc[0]

    return BeeTravel



Utilizes the previous function on every individual bee to have a full table.

In [6]:
def cleanData(data):

    activity = []
    
    allActivity = (
        data.sort_values(["tagID","datetime"])
        .groupby("tagID", group_keys = False)[data.columns]
        .apply(classifyLoc)
        .reset_index(drop=True)
    )
    
    
    df = (allActivity[allActivity['location'].isin(["Outside","Outside-Unknown"])]
        [["tagID","start","end"]]
        .rename(columns={'start':'tripStart','end':'tripEnd'})
        .copy()
    )
     
     
    df['duration'] = df['tripEnd'] - df['tripStart']

    return allActivity, df

In [7]:
def fixHourMin(duration):
    hour = int(duration.total_seconds() // 3600)
    mins = int(duration.total_seconds() // 60 % 60)
    if mins < 10:
        mins = f'0{mins}'
    return f'{hour}:{mins}'

Creates a statistical summary of an entire hive based on two tables, all activity and all flights.
Returns a table containing the statistical summary.

Does so by iterating over every beeID separately.

In [8]:
def summaryData(activity,flights):
    
    
    seen = activity.groupby("tagID").agg(
        firstSeen = ('start','min'),
        lastSeen = ('end','max')
    )
    
    seen['firstSeen'] = seen['firstSeen'].dt.floor("s")
    seen['lastSeen'] = seen['lastSeen'].dt.floor("s")
    
    indFlight = flights.groupby("tagID").agg(
        firstFlight = ('tripStart','min'),
        lastFlight = ('tripStart','max'),
        avgTrip = ('duration','mean'),
        medianTrip = ('duration','median'),
        numTrips = ('duration','size')
    )
    
    indFlight['firstFlight'] = indFlight['firstFlight'].dt.floor("s").apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    indFlight['lastFlight'] = indFlight['lastFlight'].dt.floor("s").apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    indFlight['avgTrip'] = indFlight['avgTrip'].map(fixHourMin)
    indFlight['medianTrip'] = indFlight['medianTrip'].map(fixHourMin)
    
    tpd = flights.groupby("tagID").size().rename("tripsPerDay")
    
    full = seen.join(indFlight, how="left").join(tpd, how="left")
    full["tripsPerDay"].fillna(0, inplace = True)
      
    divisor = (full['lastSeen'].dt.date - full['firstSeen'].dt.date).apply(lambda x: x.days) + 1
    full['tripsPerDay'] = full["tripsPerDay"]/divisor
    full['tripsPerDay'] = full["tripsPerDay"].apply(lambda x: x if x > 0 else None)
    
    full['firstSeen'] = full['firstSeen'].apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    full['lastSeen'] = full['lastSeen'].apply(lambda x: x.strftime('%Y-%m-%d ~ %H:%M:%S'))
    longest = flights.groupby('tagID')['duration'].idxmax()
    shortest = flights.groupby('tagID')['duration'].idxmin()
    
    long_df = flights.loc[longest, ["tagID", "tripStart", "duration"]].set_index("tagID")
    short_df = flights.loc[shortest, ["tagID", "tripStart", "duration"]].set_index("tagID")
    
    long_str = (
        long_df["tripStart"].dt.floor("s").astype(str)
        + " ~ "
        + long_df["duration"].map(fixHourMin)
    ).rename("longestFlight")
    
    short_str = (
        short_df["tripStart"].dt.floor("s").astype(str)
        + " ~ "
        + short_df["duration"].map(fixHourMin)
    ).rename("shortestFlight")
    
    full = full.join(long_str, how="left").join(short_str, how="left")
    
    beeSummary = full.rename(columns={"firstSeen":"First Seen", "lastSeen":"Last Seen","firstFlight":"First Flight",
                            "lastFlight":"Last Flight", "longestFlight":"Longest Flight","shortestFlight":"Shortest Flight",
                            "avgTrip":"Average Flight Duration", "numTrips":"# of Flights","tripsPerDay":"Average Flights per Day", "medianTrip":"Median Flight Duration"})
    beeSummary = beeSummary.reset_index()

    return beeSummary

Total summary table creation with values of all hive.

In [9]:
def makeTotalSum(flights):
    #general overview
    totalFlights = len(flights)
    flights['day'] = flights['tripStart'].apply(lambda x: x.date())
    totalPerDay = flights['day'].value_counts().mean()
    totalLength = flights['duration'].mean()
    totalLength = totalLength.total_seconds()/60

    #longest and shortest flights
    longest = flights['duration'].idxmax()
    shortest = flights['duration'].idxmin()

    longestDate = flights.loc[longest]['tripStart'].strftime('%Y-%m-%d ~ %H:%M:%S')
    longestID = flights.loc[longest]['tagID']
    longestLen = flights.loc[longest]['duration'].total_seconds()/60

    shortestDate = flights.loc[shortest]['tripStart'].strftime('%Y-%m-%d ~ %H:%M:%S')
    shortestID = flights.loc[shortest]['tagID']
    shortestLen = flights.loc[shortest]['duration'].total_seconds()/60

    #create tables
    #general info
    totalDict = {"Total Flights":totalFlights, "Avg Flights per Day":totalPerDay, "Avg Length per Day (minutes)":totalLength}
    totalSum = pd.DataFrame.from_dict(totalDict, orient='index')
    totalSum = totalSum.rename(columns={0: "Overall Statistics"})

    #info on longest/shortest trip
    tripDict = {"Classification":["Longest Flight", "Shortest Flight"],"Date":[longestDate, shortestDate], "BeeID":[longestID, shortestID], "Length (minutes)":[longestLen, shortestLen]}
    tripSum = pd.DataFrame.from_dict(tripDict)
    
    return totalSum, tripSum

Obtain sunrise and sunset table for actograph based on previously defined timezone and required days for table.

In [10]:
def getSRSS(dataset, tz, lat, lon):
    dates = dataset['start'].apply(lambda x: x.date()).sort_values()
    dates = dates.unique()
    sunrise = []
    sunset = []
    sun = Sun(lat, lon)
    for d in dates:
        dT = datetime.datetime(d.year, d.month, d.day)
        sunrise.append(sun.get_sunrise_time(dT).astimezone(tz).time())
        sunset.append(sun.get_sunset_time(dT).astimezone(tz).time())

    srss = pd.DataFrame.from_dict({"date":dates,"sunrise":sunrise,"sunset":sunset})
    return srss


Split overnight flights at midnight for plotting reasons

missing = True and timeam/timepm for when a dataset has missing values depending
on dataset requirement.

In [11]:
def separateFlights(flights,missing=False,timeam="08:00:00",timepm="18:00:00"):
    newdata = []
    
    for i in range(len(flights)):
        row = flights.iloc[i].copy()
        row['day'] = row['tripStart'].to_pydatetime().day
        day = row['day']
        month0 = row['tripStart'].to_pydatetime().month
        month1 = row['tripEnd'].to_pydatetime().month
        
        if row['tripStart'].to_pydatetime().day != row['tripEnd'].to_pydatetime().day:
            #first day
            newentry = row.copy()
            newentry['tripEnd'] = row['tripEnd'].replace(hour=23,minute=59,second=59,day=1,month=1)
            newentry['tripStart'] = row['tripStart'].replace(day=1,month=1)
            newentry['date'] = fixDate(row['tripStart'])
            newdata.append(newentry)
            #second day
            newday = row['tripEnd'].to_pydatetime().day
            newentry = row.copy()
            newentry['tripStart'] = row['tripStart'].replace(day=1,hour=0,minute=0,second=0,month=1)
            newentry['tripEnd'] = row['tripEnd'].replace(day=1,month=1)
            newentry['day'] = newday
            newentry['date'] = fixDate(row['tripEnd'])
            newdata.append(newentry)
        else:
            newentry = row.copy()
            newentry['tripStart'] = row['tripStart'].replace(day=1,month=1)
            newentry['tripEnd'] = row['tripEnd'].replace(day=1,month=1)
            newentry['date'] = fixDate(row['tripStart'])
            newdata.append(newentry)

    if missing:
        cutoffday = str(newentry['tripStart'].date())
        #Extra for missing dataset
        for i in range(len(newdata)):
            cutoffpm = datetime.strptime(cutoffday + " " + timepm,"%Y-%m-%d %H:%M:%S")
            cutoffam = datetime.strptime(cutoffday + " " + timeam,"%Y-%m-%d %H:%M:%S")
            if newdata[i]['tripEnd'] >= cutoffpm:
                newrow = newdata[i].copy()
                newdata[i]['tripEnd'] = cutoffpm
                newdata[i]['Data'] = 'Available'
                newrow['tripStart'] = cutoffpm
                newrow['Data'] = 'Missing'
                newdata.append(newrow)
            elif newdata[i]['tripStart'] <= cutoffam:
                newrow = newdata[i].copy()
                newdata[i]['tripStart'] = cutoffam
                newdata[i]['Data'] = 'Available'
                newrow['tripEnd'] = cutoffam
                newrow['Data'] = 'Missing'
                newdata.append(newrow)
            else:
                newdata[i]['Data'] = "Available"
    else:
        for i in range(len(newdata)):
            newdata[i]['Data'] = 'Available'
    
    
    flights_mod = pd.concat(newdata, axis = 1)
    flights_mod = flights_mod.transpose().reset_index()
    flights_mod = flights_mod.drop(["index"],axis=1)
    return flights_mod


Apply shapes to time graph based on sunrise/sunset dataset.

Create time plot for bee flights utilizing flights dataset.

In [12]:
#function to change hour column into hour properly

def fixTime(hr):
    if hr < 10:
        return "0" + str(hr) + ":00"
    else:
        return str(hr) + ":00"
        
#function to change date column appropriately

def fixDate(x):
    if x.day > 9 and x.month > 9:
        return f'{x.month}/{x.day}'
    elif x.day > 9 and x.month < 10:
        return f'0{x.month}/{x.day}'
    elif x.day < 10 and x.month > 9:
        return f'{x.month}/0{x.day}'
    else:
        return f'0{x.month}/0{x.day}'

Prepare all new datasets based on csv

In [13]:
allActivity, flights = cleanData(dataset)
beeSummary = summaryData(allActivity,flights)
dt = getSRSS(allActivity, tz, lat, lon)
dt['date'] = dt['date'].apply(lambda x: fixDate(x))
Summary0, Summary1 = makeTotalSum(flights)


flights['date'] = flights['tripStart'].apply(lambda x: fixDate(x))
flights['hour'] = flights['tripStart'].apply(lambda x: x.hour)
flights['day'] = flights['tripStart'].apply(lambda x: x.day)
flights_mod = separateFlights(flights)

In [14]:
flights

,tagID,tripStart,tripEnd,duration,day,date,hour
1,197,2017-06-21 08:11:24.300,2017-06-21 08:28:03.650,0 days 00:16:39.350000,21,06/21,8
3,197,2017-06-21 08:37:01.650,2017-06-21 08:58:19.050,0 days 00:21:17.400000,21,06/21,8
7,197,2017-06-21 11:36:20.200,2017-06-21 13:10:58.100,0 days 01:34:37.900000,21,06/21,11
20,197,2017-06-22 16:13:54.750,2017-06-24 09:07:51.850,1 days 16:53:57.100000,22,06/22,16
22,197,2017-06-24 10:24:33.900,2017-06-24 15:44:37.050,0 days 05:20:03.150000,24,06/24,10
...,...,...,...,...,...,...,...
1813,2487,2017-06-22 08:23:36.650,2017-06-22 09:18:19.850,0 days 00:54:43.200000,22,06/22,8
1818,2487,2017-06-22 13:10:09.450,2017-06-22 15:16:52.900,0 days 02:06:43.450000,22,06/22,13
1821,2487,2017-06-23 09:08:55.000,2017-06-23 09:38:04.300,0 days 00:29:09.300000,23,06/23,9
1825,2487,2017-06-24 08:36:48.600,2017-06-24 09:49:40.550,0 days 01:12:51.950000,24,06/24,8


Divide bees in day-focus/afternoon-focus

In [15]:
def divideBees(dataset):
    hour = dataset['hour'].unique()
    hour.sort()
    split = np.where(hour == 12)[0][0]
    final = {"morning":[],"afternoon":[],"even":[]}
    vectors = {}
    maxcount = dataset[['tagID','hour']].value_counts().max()
    divisor = 10 ** math.ceil(math.log10(maxcount))
    #print(split)
    for i in dataset['tagID'].unique():
        bee = dataset[dataset['tagID'] == i]
        data, bins = np.histogram(bee['hour'], bins=hour,density=True)
        vector = bee['hour'].value_counts().reindex(hour, fill_value=0)
        vector = vector.sort_index()
        vector = vector/divisor
        #print(i)
        #print(data)
        morning = sum(data[0:split])
        afternoon = sum(data[split:])
        if morning > 0.7:
            final['morning'].append(i)
        elif afternoon > 0.7:
            final['afternoon'].append(i)
        else:
            final['even'].append(i)

        vectors[str(i)] = vector.tolist()

    return final,vectors
        

division, vectors = divideBees(flights)

Cosine similarity for collapsed vectors

In [16]:
threshold1 = 0.8
def findSimilar(vectors, tagID):
    similars = []
    current = vectors[str(tagID)]
    for key,value in vectors.items():
        A = value
        B = current
        cosine = np.dot(A, B) / (norm(A) * norm(B))
        if cosine > threshold1:
            similars.append(key)
    return(similars)

findSimilar(vectors,1621)

['197', '508', '1237', '1361', '1621', '1780', '2487']

In [17]:
#matrix for every day where 1 = activity, 0 = no activity (matrixes_simp)
#matrix where each value is % of hour spent outside (matrixes)


def createMatrix(df):
    df = df.copy()

    df.sort_values(by='date')
    df['day'] = df['date'].astype('category').cat.codes
    df['tripStart'] = pd.to_datetime(df['tripStart'])
    df['tripEnd']   = pd.to_datetime(df['tripEnd'])
    df['start_hour'] = df['tripStart'].dt.hour
    df['end_hour'] = df['tripEnd'].dt.hour

    n_days = df['day'].nunique()

    result = {}
    result_simple = {}
    result_count = {}

    for tag, g in df.groupby('tagID'):
        matrix = np.zeros((n_days, 24))
        matrix_simple = np.zeros((n_days, 24))
        matrix_count = np.zeros((n_days, 24))

        for index, row in g.iterrows():
            matrix_simple[row.day, row.start_hour:row.end_hour+1] = 1
            matrix_count[row.day, row.start_hour] += 1
            start_h = row.tripStart.hour
            end_h   = row.tripEnd.hour

            for h in range(start_h, end_h + 1):
                hour_start = row.tripStart.floor('D') + pd.Timedelta(hours=h)
                hour_end   = hour_start + pd.Timedelta(hours=1)

                overlap = (
                    min(row.tripEnd, hour_end)
                    - max(row.tripStart, hour_start)
                ).total_seconds()

                if overlap > 0:
                    matrix[row.day, h] += overlap / 3600

                    
        result[str(tag)] = matrix
        result_simple[str(tag)] = matrix_simple
        result_count[str(tag)] = matrix_count

    return result, result_simple, result_count
            
matrixes, matrixes_simp, matrixes_count = createMatrix(flights)

bin size experimentation

In [18]:
#flatten matrix and find cosine similarity of flattened matrix (vector)
threshold2 = 0.6
flattened = {}
for k in matrixes:
    flat = matrixes[k].copy()
    flat = flat.flatten()
    flattened[k] = flat

def findSimilarMatrix(vectors, tagID):
    similars = []
    current = vectors[str(tagID)]
    for key,value in vectors.items():
        A = value
        B = current
        cosine = np.dot(A, B) / (norm(A) * norm(B))
        if cosine > threshold2:
            similars.append(key)
    return(similars)
    
findSimilarMatrix(flattened,1621)

['1361', '1621']

In [19]:
#find cosine similarity of each day's vector, and then get score from how many days are similar
def findSimilarDay(matrixes, tagID, threshold):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        A = value
        B = current
        #count of all cosine similarities
        Acount = []
        for day in range(len(A)):
            Avec = A[day]
            Bvec = B[day]
            if norm(Avec) == 0 and norm(Bvec) == 0:
                Acount.append(1)
                continue
            elif (norm(Avec) * norm(Bvec)) == 0:
                Acount.append(0)
                continue
            cosine = np.dot(Avec, Bvec) / (norm(Avec) * norm(Bvec))
            if cosine >= threshold:
                Acount.append(1)
            else:
                Acount.append(0)
        if sum(Acount)/len(Acount) >= threshold:
            similars.append(key)
    return(similars)

findSimilarDay(matrixes_simp,1621,0.7)

['1361', '1621']

In [20]:
#find cosine similarity of each day's vector, and then get score from how many days are similar
def findSimilarDay(matrixes, tagID, threshold):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        A = value
        B = current
        #count of all cosine similarities
        Acount = []
        for day in range(len(A)):
            Avec = A[day]
            Bvec = B[day]
            if norm(Avec) == 0 and norm(Bvec) == 0:
                Acount.append(1)
                continue
            elif (norm(Avec) * norm(Bvec)) == 0:
                Acount.append(0)
                continue
            cosine = np.dot(Avec, Bvec) / (norm(Avec) * norm(Bvec))
            if cosine >= threshold:
                Acount.append(1)
            else:
                Acount.append(0)
        if sum(Acount)/len(Acount) >= threshold:
            similars.append(key)
    return(similars)

findSimilarDay(matrixes,1621,0.7)

['1361', '1621']

In [21]:
def similarFroNorm(matrixes,tagID):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        dif = (current - value)
        n = norm(dif, ord="fro")
        print(key)
        print(n/norm(current,ord="fro"))

print("TIME MATRIX")
similarFroNorm(matrixes,1621)
print("ACTIVITY MATRIX")
similarFroNorm(matrixes_simp,1621)

TIME MATRIX
197
1.14135901106947
508
1.1492148004768075
515
1.248321064240831
516
0.9982112363355458
517
1.047056029599802
638
0.9301910133096223
661
1.0851031801410649
677
0.9683506832625238
930
1.010970740392928
1230
0.9169514980541443
1237
1.22627283149459
1361
1.0884964776682968
1404
1.0069826538839097
1407
1.036153225578213
1415
1.002021507016564
1607
1.057124284736037
1609
1.053216769170626
1621
0.0
1651
0.9999086837955651
1652
1.0540116520124556
1698
0.997143822781802
1722
1.1773292391693118
1780
1.0859047168319231
1786
1.066524506286424
1797
1.1319287257444481
1900
0.9994981680195144
1979
1.0000528028395235
2258
1.1529928906592364
2487
1.0668308421184136
ACTIVITY MATRIX
197
1.0339078862458586
508
1.0170952554312156
515
1.1141720290623112
516
1.0504514628777806
517
1.1596670152276025
638
0.9284766908852594
661
1.0667385033281394
677
0.909717652294684
930
0.909717652294684
1230
0.9284766908852594
1237
1.1596670152276025
1361
0.8709883407113855
1404
0.9649012813540154
1407
1.09858

In [22]:
def similarSpecNorm(matrixes,tagID):
    similars = []
    current = matrixes[str(tagID)]
    for key,value in matrixes.items():
        dif = current - value
        n = norm(dif, ord=2)
        print(key)
        print(n/norm(current, ord=2))

similarSpecNorm(matrixes,1621)

197
1.1375623059542816
508
1.0429719175906194
515
1.145845909957538
516
0.9956182975762087
517
0.9426270116094005
638
0.865573650665705
661
0.981428088808476
677
0.8952873994738805
930
0.9597841349539907
1230
0.9156596695102815
1237
1.1830430571980541
1361
1.1123697010527813
1404
0.9865892223291918
1407
0.9659577068493876
1415
0.9170145872257528
1607
0.9598800615876958
1609
0.9558309913699674
1621
0.0
1651
0.9971143132466597
1652
0.9579312690179217
1698
0.9937932977744129
1722
1.0445971909638023
1780
1.0361951846845432
1786
0.8931897020749682
1797
1.0559209499496267
1900
0.9973748284713884
1979
0.9983537593170445
2258
1.0339659987579037
2487
0.8999157857033094


In [23]:
flights['duration'].mean()

Timedelta('0 days 01:01:08.235208711')

In [24]:
flights['duration'].median()

Timedelta('0 days 00:02:58.750000')

In [25]:
flights['duration'].std()

Timedelta('0 days 03:57:49.516563168')

In [26]:
allActivity['duration'] = (allActivity['end'] - allActivity['start']).apply(lambda x: x.total_seconds())
allActivity['day'] = allActivity['start'].dt.day
allActivity['daydiff'] = (allActivity['end'].dt.hour - allActivity['start'].dt.hour)
dormant = allActivity[(allActivity['location'] == "Inside") & (allActivity['daydiff'] == 0)]
flights = allActivity[(allActivity['location'] == "Outside") & (allActivity['daydiff'] == 0)]
flights['date'] = flights['start'].apply(lambda x: fixDate(x))

In [27]:
flights

,location,start,end,age,activity,tagID,duration,day,daydiff,date
1,Outside,2017-06-21 08:11:24.300,2017-06-21 08:28:03.650,0,outside-foraging,197,999.35,21,0,06/21
3,Outside,2017-06-21 08:37:01.650,2017-06-21 08:58:19.050,0,outside-foraging,197,1277.40,21,0,06/21
31,Outside,2017-06-24 11:04:18.650,2017-06-24 11:53:37.300,3,outside-foraging,508,2958.65,24,0,06/24
41,Outside,2017-06-26 08:23:00.800,2017-06-26 08:50:56.300,4,outside-foraging,508,1675.50,26,0,06/26
58,Outside,2017-06-21 10:59:13.600,2017-06-21 10:59:44.500,0,outside-short,515,30.90,21,0,06/21
...,...,...,...,...,...,...,...,...,...,...
1794,Outside,2017-06-21 13:57:46.800,2017-06-21 13:59:52.700,0,outside-short,1979,125.90,21,0,06/21
1806,Outside,2017-06-21 16:12:11.050,2017-06-21 16:33:29.650,0,outside-foraging,2487,1278.60,21,0,06/21
1810,Outside,2017-06-22 08:02:33.400,2017-06-22 08:02:58.150,0,outside-short,2487,24.75,22,0,06/22
1821,Outside,2017-06-23 09:08:55.000,2017-06-23 09:38:04.300,1,outside-foraging,2487,1749.30,23,0,06/23


In [28]:
flights_count = flights.copy()
flights_count['half_hour'] = flights_count['start'].apply(lambda x: x.hour + round(x.minute/30)/2)
flights_count['20minute']= flights_count['start'].apply(lambda x: x.hour + math.floor((x.minute/20) * 3)/3)
flights_count['10minute']= flights_count['start'].apply(lambda x: x.hour + math.floor((x.minute/10) * 6)/6)
flights_count['5minute'] = flights_count['start'].apply(lambda x: x.hour + math.floor((x.minute/20) * 12)/12)

## FREQUENCY (D1)

Without filling empty bins

In [29]:
from scipy.spatial.distance import pdist, squareform
from scipy.stats import wasserstein_distance

In [30]:
flights_nofill = flights.groupby(['tagID','date']).size().unstack()
x = flights_nofill.apply(lambda x: [x.dropna().mean(), x.dropna().std(), stats.variation(x.dropna()), stats.skew(x.dropna())] if len(x.dropna()) > 3 and  x.dropna().std() != 0 else pd.NA, axis=1)
x = x.dropna()

In [31]:
x

tagID
516     [1.125, 0.3535533905932738, 0.2939723678960656...
517     [4.142857142857143, 2.3401261667248794, 0.5229...
638     [2.0, 0.6324555320336759, 0.28867513459481287,...
1237    [2.2857142857142856, 1.1126972805283737, 0.450...
1361    [1.4, 0.8944271909999161, 0.5714285714285716, ...
1607    [15.0, 8.847921466972585, 0.5517648452415617, ...
1609    [2.7142857142857144, 1.4960264830861911, 0.510...
1621    [2.25, 1.2583057392117916, 0.48432210483785265...
1698    [4.5, 3.7859388972001824, 0.7286042804780001, ...
1780    [3.3333333333333335, 3.8297084310253524, 1.048...
1797    [2.5, 1.760681686165901, 0.6429100507328637, 0...
1900    [2.75, 2.8722813232690143, 0.9045340337332909,...
dtype: object

In [32]:
array = np.array(x.tolist())
squareform(pdist(array))

array([[ 0.        ,  4.23578665,  2.44669095,  2.50139192,  1.0171765 ,
        16.46296489,  2.85725842,  2.17511505,  5.00774629,  4.23371469,
         2.7617449 ,  3.28661143],
       [ 4.23578665,  0.        ,  2.75092672,  2.23068669,  3.4152569 ,
        12.66197517,  1.6643453 ,  2.25718417,  1.74644858,  2.37048919,
         1.77049557,  1.83388602],
       [ 2.44669095,  2.75092672,  0.        ,  0.61268534,  1.66089694,
        15.38249217,  1.15934344,  0.95791496,  4.15991531,  3.90735994,
         1.33374879,  2.66361586],
       [ 2.50139192,  2.23068669,  0.61268534,  0.        ,  1.59923173,
        14.88906535,  0.57809661,  0.48499411,  3.56546826,  3.30665068,
         0.72896805,  2.06916287],
       [ 1.0171765 ,  3.4152569 ,  1.66089694,  1.59923173,  0.        ,
        15.85082734,  1.94666321,  1.25760049,  4.27665723,  3.54980749,
         1.80616552,  2.45637259],
       [16.46296489, 12.66197517, 15.38249217, 14.88906535, 15.85082734,
         0.        , 1

In [33]:
x = flights_nofill.apply(lambda x: [v for v in x if pd.notna(v)],axis=1)
x

tagID
197                                             [2.0]
508                                        [1.0, 1.0]
515                                   [2.0, 1.0, 1.0]
516          [1.0, 1.0, 2.0, 1.0, 1.0, 1.0, 1.0, 1.0]
517               [3.0, 7.0, 5.0, 2.0, 1.0, 7.0, 4.0]
638                    [2.0, 2.0, 3.0, 2.0, 2.0, 1.0]
677                                             [1.0]
930                                             [1.0]
1230                                [38.0, 1.0, 18.0]
1237              [3.0, 1.0, 3.0, 1.0, 2.0, 2.0, 4.0]
1361                        [3.0, 1.0, 1.0, 1.0, 1.0]
1404                                            [3.0]
1407                                  [9.0, 9.0, 1.0]
1607    [23.0, 6.0, 23.0, 4.0, 5.0, 21.0, 14.0, 24.0]
1609              [1.0, 1.0, 5.0, 3.0, 4.0, 2.0, 3.0]
1621                             [2.0, 4.0, 1.0, 2.0]
1651                                            [2.0]
1652              [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
1698                  

In [34]:
arrays = [wasserstein_distance(i,j) for i in x for j in x]
arr = np.array(arrays).reshape((len(x),len(x)))
arr

array([[ 0.        ,  1.        ,  0.66666667,  0.875     ,  2.42857143,
         0.33333333,  1.        ,  1.        , 17.66666667,  0.85714286,
         1.        ,  1.        ,  5.        , 13.        ,  1.28571429,
         0.75      ,  0.        ,  1.        ,  2.5       ,  1.        ,
         2.        ,  1.        ,  1.5       ,  1.75      ,  1.        ,
         1.        ],
       [ 1.        ,  0.        ,  0.33333333,  0.125     ,  3.14285714,
         1.        ,  0.        ,  0.        , 18.        ,  1.28571429,
         0.4       ,  2.        ,  5.33333333, 14.        ,  1.71428571,
         1.25      ,  1.        ,  0.        ,  3.5       ,  0.        ,
         2.33333333,  0.        ,  1.5       ,  1.75      ,  0.        ,
         0.        ],
       [ 0.66666667,  0.33333333,  0.        ,  0.20833333,  2.80952381,
         0.66666667,  0.33333333,  0.33333333, 17.66666667,  0.95238095,
         0.33333333,  1.66666667,  5.        , 13.66666667,  1.38095238,
       

Filling empty bins with 0

In [35]:
flights_freq = flights.groupby(['tagID','date']).size().unstack(fill_value=0)
flights_freq

date,06/21,06/22,06/23,06/24,06/25,06/26,06/27,06/28
tagID,,,,,,,,
197,2,0,0,0,0,0,0,0
508,0,0,0,1,0,1,0,0
515,2,1,1,0,0,0,0,0
516,1,1,2,1,1,1,1,1
517,3,7,5,2,1,7,4,0
638,2,2,3,2,0,2,1,0
677,1,0,0,0,0,0,0,0
930,0,1,0,0,0,0,0,0
1230,38,1,18,0,0,0,0,0


In [36]:
x = flights_freq.apply(lambda x: [x.mean(), x.std(), stats.variation(x), stats.skew(x)], axis=1)
x

tagID
197     [0.25, 0.7071067811865476, 2.6457513110645907,...
508     [0.25, 0.4629100498862757, 1.7320508075688772,...
515     [0.5, 0.7559289460184544, 1.4142135623730951, ...
516     [1.125, 0.3535533905932738, 0.2939723678960656...
517     [3.625, 2.615202805574687, 0.6748408893372733,...
638     [1.5, 1.0690449676496976, 0.6666666666666666, ...
677     [0.125, 0.3535533905932738, 2.6457513110645907...
930     [0.125, 0.3535533905932738, 2.6457513110645907...
1230    [7.125, 13.953366209321269, 1.8318847630690818...
1237    [2.0, 1.3093073414159542, 0.6123724356957945, ...
1361    [0.875, 0.9910312089651149, 1.0594569267279519...
1404    [0.375, 1.0606601717798212, 2.6457513110645907...
1407    [2.375, 4.103569874424672, 1.616226582398325, ...
1607    [15.0, 8.847921466972585, 0.5517648452415617, ...
1609    [2.375, 1.685018016012207, 0.6636589585746574,...
1621    [1.125, 1.4577379737113252, 1.2120791238484125...
1651    [0.25, 0.7071067811865476, 2.6457513110645907,...
1652    

In [37]:
array = np.array(x.tolist())
squareform(pdist(array))

array([[ 0.        ,  1.4606306 ,  1.74319351,  2.53406574,  4.86884438,
         3.5488868 ,  0.375     ,  0.375     , 14.96065141,  3.56407524,
         2.03616601,  0.375     ,  4.28983072, 17.16138063,  3.7350792 ,
         2.25904631,  0.        ,  5.12151449,  3.67233604,  1.4606306 ,
         3.89599641,  1.4606306 ,  3.11702993,  2.29753038,  0.375     ,
         2.81844636],
       [ 1.4606306 ,  0.        ,  0.50815763,  2.02104295,  4.27753201,
         2.32484981,  1.44961843,  1.44961843, 15.14851056,  2.52307379,
         1.0598459 ,  1.56420585,  4.21710592, 17.06491491,  2.86200945,
         1.43696215,  1.4606306 ,  3.73491011,  3.63466821,  0.        ,
         3.94135464,  0.        ,  2.36577645,  2.34739283,  1.44961843,
         1.3917112 ],
       [ 1.74319351,  0.50815763,  0.        ,  1.80682766,  3.83827645,
         1.92820853,  1.81007489,  1.81007489, 14.78315601,  2.07946552,
         0.58212379,  1.75565547,  3.84293544, 16.67836542,  2.40824067,
       

In [38]:
x = flights_freq.apply(lambda x: x.tolist(), axis=1)
x

tagID
197          [2, 0, 0, 0, 0, 0, 0, 0]
508          [0, 0, 0, 1, 0, 1, 0, 0]
515          [2, 1, 1, 0, 0, 0, 0, 0]
516          [1, 1, 2, 1, 1, 1, 1, 1]
517          [3, 7, 5, 2, 1, 7, 4, 0]
638          [2, 2, 3, 2, 0, 2, 1, 0]
677          [1, 0, 0, 0, 0, 0, 0, 0]
930          [0, 1, 0, 0, 0, 0, 0, 0]
1230       [38, 1, 18, 0, 0, 0, 0, 0]
1237         [3, 1, 3, 1, 2, 2, 4, 0]
1361         [3, 0, 1, 1, 1, 0, 1, 0]
1404         [3, 0, 0, 0, 0, 0, 0, 0]
1407         [9, 9, 1, 0, 0, 0, 0, 0]
1607    [23, 6, 23, 4, 5, 21, 14, 24]
1609         [1, 1, 5, 3, 4, 2, 3, 0]
1621         [2, 0, 4, 1, 2, 0, 0, 0]
1651         [2, 0, 0, 0, 0, 0, 0, 0]
1652         [1, 1, 1, 1, 1, 1, 1, 0]
1698        [10, 2, 2, 4, 0, 0, 0, 0]
1722         [0, 1, 0, 0, 0, 0, 1, 0]
1780        [1, 2, 2, 1, 0, 11, 0, 3]
1786         [1, 1, 0, 0, 0, 0, 0, 0]
1797         [4, 1, 0, 0, 5, 1, 3, 1]
1900         [1, 1, 2, 7, 0, 0, 0, 0]
1979         [1, 0, 0, 0, 0, 0, 0, 0]
2487         [1, 1, 1, 1, 0, 0, 0, 0]
dtype:

Full matrix

In [39]:
arrays = [wasserstein_distance(i,j) for i in x for j in x]
arr = np.array(arrays).reshape((len(x),len(x)))

In [40]:
arr

array([[ 0.   ,  0.25 ,  0.25 ,  0.875,  3.375,  1.25 ,  0.125,  0.125,
         6.875,  1.75 ,  0.625,  0.125,  2.125, 14.75 ,  2.125,  0.875,
         0.   ,  0.875,  2.   ,  0.25 ,  2.25 ,  0.25 ,  1.625,  1.125,
         0.125,  0.5  ],
       [ 0.25 ,  0.   ,  0.25 ,  0.875,  3.375,  1.25 ,  0.125,  0.125,
         6.875,  1.75 ,  0.625,  0.375,  2.125, 14.75 ,  2.125,  0.875,
         0.25 ,  0.625,  2.   ,  0.   ,  2.25 ,  0.   ,  1.625,  1.125,
         0.125,  0.25 ],
       [ 0.25 ,  0.25 ,  0.   ,  0.625,  3.125,  1.   ,  0.375,  0.375,
         6.625,  1.5  ,  0.375,  0.375,  1.875, 14.5  ,  1.875,  0.625,
         0.25 ,  0.625,  1.75 ,  0.25 ,  2.   ,  0.25 ,  1.375,  0.875,
         0.375,  0.25 ],
       [ 0.875,  0.875,  0.625,  0.   ,  2.75 ,  0.875,  1.   ,  1.   ,
         7.25 ,  1.125,  0.5  ,  1.   ,  2.5  , 13.875,  1.5  ,  1.   ,
         0.875,  0.25 ,  2.125,  0.875,  1.875,  0.875,  1.25 ,  1.25 ,
         1.   ,  0.625],
       [ 3.375,  3.375,  3.125,  2.7

In [41]:
squareform(arr)

array([ 0.25 ,  0.25 ,  0.875,  3.375,  1.25 ,  0.125,  0.125,  6.875,
        1.75 ,  0.625,  0.125,  2.125, 14.75 ,  2.125,  0.875,  0.   ,
        0.875,  2.   ,  0.25 ,  2.25 ,  0.25 ,  1.625,  1.125,  0.125,
        0.5  ,  0.25 ,  0.875,  3.375,  1.25 ,  0.125,  0.125,  6.875,
        1.75 ,  0.625,  0.375,  2.125, 14.75 ,  2.125,  0.875,  0.25 ,
        0.625,  2.   ,  0.   ,  2.25 ,  0.   ,  1.625,  1.125,  0.125,
        0.25 ,  0.625,  3.125,  1.   ,  0.375,  0.375,  6.625,  1.5  ,
        0.375,  0.375,  1.875, 14.5  ,  1.875,  0.625,  0.25 ,  0.625,
        1.75 ,  0.25 ,  2.   ,  0.25 ,  1.375,  0.875,  0.375,  0.25 ,
        2.75 ,  0.875,  1.   ,  1.   ,  7.25 ,  1.125,  0.5  ,  1.   ,
        2.5  , 13.875,  1.5  ,  1.   ,  0.875,  0.25 ,  2.125,  0.875,
        1.875,  0.875,  1.25 ,  1.25 ,  1.   ,  0.625,  2.125,  3.5  ,
        3.5  ,  7.   ,  1.625,  2.75 ,  3.25 ,  2.25 , 11.375,  1.25 ,
        2.5  ,  3.375,  2.75 ,  2.125,  3.375,  2.125,  3.375,  1.75 ,
      

In [42]:
np.allclose(arr, arr.T)

True

### DURATION SIMILARITY (D2)

#### TIME BETWEEN FLIGHTS

In [43]:
dormant

,location,start,end,age,activity,tagID,duration,day,daydiff
0,Inside,2017-06-21 08:07:56.250,2017-06-21 08:11:24.300,0,inside,197,208.05,21,0
2,Inside,2017-06-21 08:28:03.650,2017-06-21 08:37:01.650,0,inside,197,538.00,21,0
40,Inside,2017-06-26 08:00:17.300,2017-06-26 08:23:00.800,4,inside,508,1363.50,26,0
47,Inside,2017-06-27 09:35:44.700,2017-06-27 09:37:19.850,5,inside,508,95.15,27,0
49,Inside,2017-06-27 10:10:18.650,2017-06-27 10:11:22.450,5,inside,508,63.80,27,0
...,...,...,...,...,...,...,...,...,...
1800,Inside,2017-06-21 09:32:24.700,2017-06-21 09:55:27.800,0,inside,2487,1383.10,21,0
1802,Inside,2017-06-21 10:32:23.300,2017-06-21 10:43:30.450,0,inside,2487,667.15,21,0
1809,Inside,2017-06-22 08:00:57.650,2017-06-22 08:02:33.400,0,inside,2487,95.75,22,0
1812,Inside,2017-06-22 08:03:39.250,2017-06-22 08:23:36.650,0,inside,2487,1197.40,22,0


In [44]:
dormant_times = dormant.groupby(['tagID'])['duration'].apply(list)
dormant_times

tagID
197                                       [208.05, 538.0]
508          [1363.5, 95.15, 63.8, 152.3, 2832.1, 175.45]
515                         [455.35, 261.8, 18.65, 14.35]
517     [344.35, 351.2, 325.0, 515.1, 412.0, 370.0, 70...
638     [750.0, 369.25, 302.4, 376.7, 290.8, 292.35, 2...
661                                                [69.3]
677                               [142.1, 149.45, 1024.9]
1230    [426.4, 48.4, 59.65, 42.5, 66.05, 51.7, 42.2, ...
1237    [824.9, 2178.1, 1051.2, 173.7, 498.05, 165.7, ...
1361    [85.35, 136.8, 273.0, 196.7, 367.35, 544.55, 4...
1404                                              [458.4]
1407    [1281.45, 51.15, 1090.55, 2116.55, 440.3, 113....
1415                                     [672.9, 1953.85]
1607    [752.8, 11.4, 286.45, 22.75, 225.4, 12.5, 650....
1609    [474.1, 307.45, 225.4, 206.5, 156.2, 100.55, 1...
1621    [623.0, 264.85, 873.4, 2718.85, 499.55, 742.15...
1651                                   [1091.35, 1991.75]
1652    

In [45]:
arrays = [wasserstein_distance(i,j) for i in dormant_times for j in dormant_times]
arr = np.array(arrays).reshape((len(dormant_times),len(dormant_times)))
squareform(arr)

array([ 632.50833333,  185.4875    ,  236.37205882, 6408.33928571,
        303.725     ,  258.80833333,  338.4       ,  349.18333333,
        112.92916667,  164.975     ,  407.65625   ,  940.35      ,
        324.36536697,  137.91111111,  415.74558824, 1168.525     ,
        179.85357143,  205.45147059,  493.6375    ,  615.29852941,
         85.325     ,  393.40166667,  343.26666667,  557.555     ,
        621.62916667,  502.49411765, 6382.9452381 ,  712.91666667,
        383.31666667,  659.73981481,  305.00277778,  567.32916667,
        770.95      ,  300.82291667,  825.74166667,  606.99067278,
        542.26944444,  402.08235294, 1041.28333333,  511.4       ,
        566.87254902,  388.10416667,  225.7254902 ,  673.2       ,
        368.16333333,  374.15833333,  632.57      ,  344.13014706,
       6510.27678571,  171.0375    ,  288.72916667,  169.07546296,
        472.5875    ,   90.29166667,  270.8625    ,  347.65625   ,
       1125.8375    ,  160.76135321,  159.48194444,  580.66838

In [46]:
np.allclose(arr, arr.T)

True

#### DURATION OF FLIGHTS

In [47]:
active_times = flights.groupby(['tagID'])['duration'].apply(list)
active_times

tagID
197                                      [999.35, 1277.4]
508                                     [2958.65, 1675.5]
515                             [30.9, 88.2, 170.4, 61.4]
516     [71.2, 79.4, 118.0, 55.0, 27.25, 38.85, 60.6, ...
517     [77.0, 1031.85, 1829.0, 49.2, 472.15, 713.2, 7...
638     [1948.55, 1148.3, 853.4, 2086.4, 703.45, 2117....
677                                              [1996.2]
930                                             [86612.8]
1230    [33.8, 44.6, 204.95, 31.5, 27.75, 18.2, 98.15,...
1237    [767.2, 642.3, 170.35, 2405.9, 970.0, 1047.9, ...
1361    [1413.85, 13.0, 571.8, 33.4, 1245.6, 1198.6, 5...
1404                              [146.75, 889.75, 962.6]
1407    [75.85, 639.75, 470.85, 104.1, 97.65, 10.2, 69...
1607    [19.8, 13.75, 26.45, 32.75, 23.3, 20.1, 26.6, ...
1609    [24.1, 738.75, 15.15, 24.35, 115.2, 2876.45, 1...
1621    [854.4, 276.6, 30.6, 1730.8, 13.15, 1671.6, 15...
1651                                      [127.05, 163.5]
1652    

In [48]:
arrays = [wasserstein_distance(i,j) for i in active_times for j in active_times]
arr = np.array(arrays).reshape((len(active_times),len(active_times)))
squareform(arr)

array([1.17870000e+03, 1.05065000e+03, 1.07880278e+03, 6.00937069e+02,
       5.35391667e+02, 8.57825000e+02, 8.54744250e+04, 1.07661272e+03,
       5.36987500e+02, 4.57803571e+02, 4.72008333e+02, 9.30946053e+02,
       1.01262792e+03, 6.99922368e+02, 5.69730556e+02, 9.93100000e+02,
       4.37189286e+02, 1.10498056e+03, 6.51025000e+02, 1.05087000e+03,
       5.52750000e+01, 9.00698333e+02, 1.09017500e+03, 1.01247500e+03,
       5.92000000e+02, 2.22935000e+03, 2.25750278e+03, 1.74159569e+03,
       1.13980833e+03, 6.41575000e+02, 8.42957250e+04, 2.25531272e+03,
       1.14816250e+03, 1.59751786e+03, 1.65070833e+03, 2.10964605e+03,
       2.17214292e+03, 1.66714342e+03, 1.48444167e+03, 2.17180000e+03,
       1.09173929e+03, 2.28368056e+03, 8.79625000e+02, 2.22957000e+03,
       1.16997500e+03, 1.74283167e+03, 2.26887500e+03, 2.19117500e+03,
       1.53415000e+03, 3.00055556e+01, 4.88319828e+02, 1.09182500e+03,
       1.90847500e+03, 8.65250750e+04, 3.95416667e+01, 1.09531250e+03,
      

In [49]:
np.allclose(arr, arr.T)

True

### SCHEDULE SIMILARITY (D3)

Bin calculation

In [50]:
active_times

tagID
197                                      [999.35, 1277.4]
508                                     [2958.65, 1675.5]
515                             [30.9, 88.2, 170.4, 61.4]
516     [71.2, 79.4, 118.0, 55.0, 27.25, 38.85, 60.6, ...
517     [77.0, 1031.85, 1829.0, 49.2, 472.15, 713.2, 7...
638     [1948.55, 1148.3, 853.4, 2086.4, 703.45, 2117....
677                                              [1996.2]
930                                             [86612.8]
1230    [33.8, 44.6, 204.95, 31.5, 27.75, 18.2, 98.15,...
1237    [767.2, 642.3, 170.35, 2405.9, 970.0, 1047.9, ...
1361    [1413.85, 13.0, 571.8, 33.4, 1245.6, 1198.6, 5...
1404                              [146.75, 889.75, 962.6]
1407    [75.85, 639.75, 470.85, 104.1, 97.65, 10.2, 69...
1607    [19.8, 13.75, 26.45, 32.75, 23.3, 20.1, 26.6, ...
1609    [24.1, 738.75, 15.15, 24.35, 115.2, 2876.45, 1...
1621    [854.4, 276.6, 30.6, 1730.8, 13.15, 1671.6, 15...
1651                                      [127.05, 163.5]
1652    

In [51]:
bins = pd.Series([x for xs in active_times for x in xs]).apply(
    lambda x: 5 if x <= 300 else
    10 if x <= 600 else
    15 if x <= 900 else
    20 if x <= 1200 else
    30 if x <= 1800 else
    60
)

bins

0      20
1      30
2      60
3      30
4       5
       ..
387     5
388    30
389     5
390    30
391     5
Length: 392, dtype: int64

In [52]:
bins.groupby(bins).size()

5     286
10     13
15     30
20     23
30     23
60     17
dtype: int64

Using 5 minute bins

In [53]:
flights_count = flights_count.sort_values(['day','5minute'])
count = flights_count.groupby(['tagID','date','5minute']).size()
idx = pd.MultiIndex.from_product([flights_count.tagID.unique(), flights_count.day.unique(), flights_count['20minute'].unique()])
count = count.reindex(idx).fillna(0)
count
#unstacked = count.unstack(['day','20minute'], fill_value=0)
#unstacked_dict = unstacked.groupby(level=0).apply(lambda df: df.xs(df.name).to_dict()).to_dict()

516  21  8.000000     0.0
         8.333333     0.0
         9.000000     0.0
         9.333333     0.0
         9.666667     0.0
                     ... 
508  28  18.333333    0.0
         18.666667    0.0
         19.333333    0.0
         19.666667    0.0
         8.666667     0.0
Length: 7280, dtype: float64

In [54]:
result = {
    tag: {
        day: g.values.tolist()
        for day, g in tag_group.groupby(level=1)
    }
    for tag, tag_group in count.groupby(level=0)
}
result

{197: {21: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  22: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  23: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0],
  24: [0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,
   0.0,


In [55]:
def compareScheduleDaytoDay(tag1, tag2, result):
    bee1 = result[tag1]
    bee2 = result[tag2]
    values = []
    for (k,v), (k2,v2) in zip(bee1.items(), bee2.items()):
        cosine = np.dot(v, v2) / (norm(v) * norm(v2))
        values.append(cosine)
    return np.nan_to_num(values)

compareScheduleDaytoDay(1621,1361,result)

C:\Users\kashi\AppData\Local\Temp\ipykernel_85652\371221653.py:6: RuntimeWarning: invalid value encountered in scalar divide
  cosine = np.dot(v, v2) / (norm(v) * norm(v2))


array([0., 0., 0., 0., 0., 0., 0., 0.])

In [56]:
result_mat = {}
for key,val in result.items():
    mat = []
    for day, lst in val.items():
        mat.append(lst)
    npmat = np.array([np.array(xi) for xi in mat])
    result_mat[key] = npmat

result_mat

{197: array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         

In [57]:
def compareMatrixes(tag1,tag2,result):
    bee1 = result[tag1]
    bee2 = result[tag2]
    return cosine_similarity(bee1,bee2)

np.triu(compareMatrixes(1621,1361,result_mat))

array([[0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0., 0., 0., 0.]])

### TREND SIMILARITY (D4)

#### FLIGHT FREQUENCY

In [97]:
from sklearn.linear_model import LinearRegression 

In [98]:
flights_nofill

date,06/21,06/22,06/23,06/24,06/25,06/26,06/27,06/28
tagID,,,,,,,,
197,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
508,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN
515,2.0,1.0,1.0,NaN,NaN,NaN,NaN,NaN
516,1.0,1.0,2.0,1.0,1.0,1.0,1.0,1.0
517,3.0,7.0,5.0,2.0,1.0,7.0,4.0,NaN
638,2.0,2.0,3.0,2.0,NaN,2.0,1.0,NaN
677,1.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
930,NaN,1.0,NaN,NaN,NaN,NaN,NaN,NaN
1230,38.0,1.0,18.0,NaN,NaN,NaN,NaN,NaN


In [111]:
reg = flights_nofill.apply(
    lambda x: [(d,v) for d,v in x.items() if pd.notna(v)],
    axis=1
)
reg

tagID
197                                        [(06/21, 2.0)]
508                          [(06/24, 1.0), (06/26, 1.0)]
515            [(06/21, 2.0), (06/22, 1.0), (06/23, 1.0)]
516     [(06/21, 1.0), (06/22, 1.0), (06/23, 2.0), (06...
517     [(06/21, 3.0), (06/22, 7.0), (06/23, 5.0), (06...
638     [(06/21, 2.0), (06/22, 2.0), (06/23, 3.0), (06...
677                                        [(06/21, 1.0)]
930                                        [(06/22, 1.0)]
1230         [(06/21, 38.0), (06/22, 1.0), (06/23, 18.0)]
1237    [(06/21, 3.0), (06/22, 1.0), (06/23, 3.0), (06...
1361    [(06/21, 3.0), (06/23, 1.0), (06/24, 1.0), (06...
1404                                       [(06/21, 3.0)]
1407           [(06/21, 9.0), (06/22, 9.0), (06/23, 1.0)]
1607    [(06/21, 23.0), (06/22, 6.0), (06/23, 23.0), (...
1609    [(06/21, 1.0), (06/22, 1.0), (06/23, 5.0), (06...
1621    [(06/21, 2.0), (06/23, 4.0), (06/24, 1.0), (06...
1651                                       [(06/21, 2.0)]
1652    

In [112]:
days = pd.Series(np.sort(flights['date'].unique()))
day_map = dict(zip(days, days.astype('category').cat.codes))
day_map

{'06/21': 0,
 '06/22': 1,
 '06/23': 2,
 '06/24': 3,
 '06/25': 4,
 '06/26': 5,
 '06/27': 6,
 '06/28': 7}

In [113]:
regs = {}
for index, val in reg.items():
    x = []
    y = []
    for v in val:
        x.append(day_map[v[0]])
        y.append(v[1])
    x = np.array(x).reshape([-1,1])
    linreg = LinearRegression().fit(x, y)
    slope = linreg.coef_
    regs[index] = slope

regs = pd.Series(regs)
regs

197                      [0.0]
508                      [0.0]
515      [-0.4999999999999998]
516     [-0.03571428571428572]
517     [-0.03571428571428572]
638     [-0.14906832298136646]
677                      [0.0]
930                      [0.0]
1230      [-9.999999999999995]
1237     [0.14285714285714282]
1361    [-0.29999999999999993]
1404                     [0.0]
1407     [-3.9999999999999987]
1607      [0.5000000000000001]
1609     [0.24999999999999992]
1621    [-0.14285714285714288]
1651                     [0.0]
1652                     [0.0]
1698     [-1.7999999999999998]
1722                     [0.0]
1780       [0.735294117647059]
1786                     [0.0]
1797     [-0.1673819742489271]
1900                     [1.9]
1979                     [0.0]
2487                     [0.0]
dtype: object

In [114]:
arrays = [abs(i-j) for index,i in regs.items() for index2, j in regs.items()]
arr = np.array(arrays).reshape((len(regs),len(regs)))
squareform(arr)

array([0.00000000e+00, 5.00000000e-01, 3.57142857e-02, 3.57142857e-02,
       1.49068323e-01, 0.00000000e+00, 0.00000000e+00, 1.00000000e+01,
       1.42857143e-01, 3.00000000e-01, 0.00000000e+00, 4.00000000e+00,
       5.00000000e-01, 2.50000000e-01, 1.42857143e-01, 0.00000000e+00,
       0.00000000e+00, 1.80000000e+00, 0.00000000e+00, 7.35294118e-01,
       0.00000000e+00, 1.67381974e-01, 1.90000000e+00, 0.00000000e+00,
       0.00000000e+00, 5.00000000e-01, 3.57142857e-02, 3.57142857e-02,
       1.49068323e-01, 0.00000000e+00, 0.00000000e+00, 1.00000000e+01,
       1.42857143e-01, 3.00000000e-01, 0.00000000e+00, 4.00000000e+00,
       5.00000000e-01, 2.50000000e-01, 1.42857143e-01, 0.00000000e+00,
       0.00000000e+00, 1.80000000e+00, 0.00000000e+00, 7.35294118e-01,
       0.00000000e+00, 1.67381974e-01, 1.90000000e+00, 0.00000000e+00,
       0.00000000e+00, 4.64285714e-01, 4.64285714e-01, 3.50931677e-01,
       5.00000000e-01, 5.00000000e-01, 9.50000000e+00, 6.42857143e-01,
      

#### FLIGHT DURATION

In [115]:
dur_nofill = flights.groupby(['tagID','date'])['duration'].agg("mean").unstack()
regDur = dur_nofill.apply(
    lambda x: [(d,v) for d,v in x.items() if pd.notna(v)],
    axis=1
)

In [116]:
regsDur = {}
for index, val in regDur.items():
    x = []
    y = []
    for v in val:
        x.append(day_map[v[0]])
        y.append(v[1])
    x = np.array(x).reshape([-1,1])
    linreg = LinearRegression().fit(x, y)
    slope = linreg.coef_
    regsDur[index] = slope

regsDur = pd.Series(regsDur)
regsDur

197                      [0.0]
508       [-641.5749999999998]
515       [0.9249999999999943]
516       [-5.449404761904763]
517       [-86.08548469387755]
638      [-116.04073498964804]
677                      [0.0]
930                      [0.0]
1230       [4.001096491228061]
1237      [-33.52113095238094]
1361      [42.425000000000004]
1404                     [0.0]
1407       [368.9472222222222]
1607      [-23.78783110889284]
1609      [35.225089285714276]
1621       [98.90249999999997]
1651                     [0.0]
1652     [-144.67857142857144]
1698                   [9.942]
1722                 [-316.02]
1780    [-0.23179590017825386]
1786     [-388.59999999999997]
1797       [327.8686945636625]
1900     [-11.908928571428572]
1979                     [0.0]
2487     [-187.40999999999997]
dtype: object

In [117]:
arrays = [abs(i-j) for index,i in regsDur.items() for index2, j in regsDur.items()]
arr = np.array(arrays).reshape((len(regsDur),len(regsDur)))
squareform(arr)

array([6.41575000e+02, 9.25000000e-01, 5.44940476e+00, 8.60854847e+01,
       1.16040735e+02, 0.00000000e+00, 0.00000000e+00, 4.00109649e+00,
       3.35211310e+01, 4.24250000e+01, 0.00000000e+00, 3.68947222e+02,
       2.37878311e+01, 3.52250893e+01, 9.89025000e+01, 0.00000000e+00,
       1.44678571e+02, 9.94200000e+00, 3.16020000e+02, 2.31795900e-01,
       3.88600000e+02, 3.27868695e+02, 1.19089286e+01, 0.00000000e+00,
       1.87410000e+02, 6.42500000e+02, 6.36125595e+02, 5.55489515e+02,
       5.25534265e+02, 6.41575000e+02, 6.41575000e+02, 6.45576096e+02,
       6.08053869e+02, 6.84000000e+02, 6.41575000e+02, 1.01052222e+03,
       6.17787169e+02, 6.76800089e+02, 7.40477500e+02, 6.41575000e+02,
       4.96896429e+02, 6.51517000e+02, 3.25555000e+02, 6.41343204e+02,
       2.52975000e+02, 9.69443695e+02, 6.29666071e+02, 6.41575000e+02,
       4.54165000e+02, 6.37440476e+00, 8.70104847e+01, 1.16965735e+02,
       9.25000000e-01, 9.25000000e-01, 3.07609649e+00, 3.44461310e+01,
      

#### DIFFERENCES

In [118]:
CoeffDif = abs(regsDur - regs)
CoeffDif

197                    [0.0]
508      [641.5749999999998]
515      [1.424999999999994]
516      [5.413690476190477]
517      [86.04977040816325]
638     [115.89166666666668]
677                    [0.0]
930                    [0.0]
1230    [14.001096491228056]
1237     [33.66398809523809]
1361                [42.725]
1404                   [0.0]
1407     [372.9472222222222]
1607     [24.28783110889284]
1609    [34.975089285714276]
1621     [99.04535714285711]
1651                   [0.0]
1652    [144.67857142857144]
1698                [11.742]
1722                [316.02]
1780    [0.9670900178253128]
1786    [388.59999999999997]
1797     [328.0360765379114]
1900    [13.808928571428572]
1979                   [0.0]
2487    [187.40999999999997]
dtype: object

#### MULTIVARIATE

In [133]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.multivariate.multivariate_ols import _MultivariateOLS

In [168]:
for i in range(len(regDur)):
    x = []
    y = []
    y2 = []
    for j in range(len(regDur.iloc[i])):     
        x.append(day_map[regDur.iloc[i][j][0]])
        y.append(regDur.iloc[i][j][1])
        y2.append(reg.iloc[i][j][1])
    if len(x) < 2:
        continue
    df = pd.DataFrame({
        'x':x,
        'y1':y,
        'y2':y2
    })
    Y = df[['y1','y2']]
    #X = sm.add_constant(df[['x']])
    X = np.array(df['x']).reshape(-1,1)
    model = _MultivariateOLS(Y, X)
    result = model.fit()
    "print(result.mv_test((('x', [[0, 1]]), None)))
    


AttributeError: '_MultivariateOLS' object has no attribute 'params'